# UK Biobank — Multimodal Aging Clock: Data Characterisation

This notebook covers the full exploratory data analysis (EDA) for a multimodal
biological aging clock trained on UK Biobank data.

**Modalities included:**
- Population characteristics (age target, sex)
- Physical measures (anthropometry, body composition, blood pressure)
- Blood biochemistry
- Blood count (full blood count)
- Metabolomics (NMR, Nightingale panel)
- Proteomics (Olink Explore)

**Sections:**
1. Setup & configuration
2. Data loading and initial inspection
3. Statistical summaries and distributions
4. Missing value analysis
5. Cross-modality overlap
6. Feature correlation analysis
7. Data quality assessment
8. Summary


## 1. Setup & Configuration

In [ ]:
import re
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.cluster.hierarchy as sch
from scipy.stats import gaussian_kde

warnings.filterwarnings("ignore")

try:
    from upsetplot import UpSet, from_memberships
    HAS_UPSET = True
except ImportError:
    HAS_UPSET = False
    print("[INFO] upsetplot not installed — UpSet plot will be skipped.")
    print("       Install with: pip install upsetplot")

# ── SET THIS TO YOUR DATA ROOT ────────────────────────────────────────────────
DATA_ROOT = Path("/superscratch/dkolbe/aDNA/UKB/data")  # set via --data_root when generating
OUT_DIR   = Path("./notebook_figures")
OUT_DIR.mkdir(exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

print(f"Data root : {DATA_ROOT.resolve()}")
print(f"Exists    : {DATA_ROOT.exists()}")


In [ ]:
# ── Styling ───────────────────────────────────────────────────────────────────
PALETTE = {
    "popchar":           "#4C72B0",
    "physical_measures": "#DD8452",
    "blood_biochemistry":"#55A868",
    "blood_count":       "#C44E52",
    "metabolomics":      "#8172B2",
    "olink":             "#937860",
}
LABELS = {
    "popchar":           "Population Characteristics",
    "physical_measures": "Physical Measures",
    "blood_biochemistry":"Blood Biochemistry",
    "blood_count":       "Blood Count",
    "metabolomics":      "Metabolomics (NMR)",
    "olink":             "Proteomics (Olink)",
}

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        11,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "grid.linestyle":   "--",
    "figure.dpi":       120,
    "savefig.dpi":      200,
    "savefig.bbox":     "tight",
})

# ── File manifest ─────────────────────────────────────────────────────────────
MODALITIES = {
    "popchar":            {"subdir": "1_phenotypic",    "data": "popchar_data.tsv",               "fmt": "tsv"},
    "physical_measures":  {"subdir": "1_phenotypic",    "data": "Physical_measures_data.parquet", "fmt": "parquet"},
    "blood_biochemistry": {"subdir": "2_blood_measures","data": "BloodBiochemistry_data.tsv",     "fmt": "tsv"},
    "blood_count":        {"subdir": "2_blood_measures","data": "BloodCount_data.tsv",            "fmt": "tsv"},
    "metabolomics":       {"subdir": "2_blood_measures","data": "metabolomics_data.tsv",          "fmt": "tsv"},
    "olink":              {"subdir": "2_blood_measures","data": "olink_data.tsv",                 "fmt": "tsv"},
}

# Physical measures chars (from Physical_measures_chars.json)
PHYS_FIELD_NAMES = {
    21: "Weight method", 36: "Blood pressure device ID",
    37: "BP manual sphygmomanometer device ID", 39: "Height measure device ID",
    40: "Manual scales device ID", 41: "Seating box device ID",
    43: "Impedance device ID", 44: "Tape measure device ID",
    48: "Waist circumference", 49: "Hip circumference",
    50: "Standing height", 51: "Seated height",
    93: "Systolic BP (manual)", 94: "Diastolic BP (manual)",
    95: "Pulse rate (BP measurement)", 96: "Time since interview start",
    102: "Pulse rate (automated)", 3077: "Seating box height",
    3160: "Weight (manual entry)", 4079: "Diastolic BP (automated)",
    4080: "Systolic BP (automated)", 4081: "Method of measuring BP",
    12143: "Weight (pre-imaging)", 12144: "Height",
    20015: "Sitting height", 20041: "Reason for skipping weight",
    20046: "Reason for skipping hip", 20047: "Reason for skipping height",
    20048: "Reason for skipping sitting height",
    21001: "BMI", 21002: "Weight", 23098: "Weight",
    23099: "Body fat %", 23100: "Whole body fat mass",
    23101: "Whole body fat-free mass", 23102: "Whole body water mass",
    23104: "BMI", 23106: "Impedance (whole body)",
    23107: "Impedance leg (R)", 23108: "Impedance leg (L)",
    23109: "Impedance arm (R)", 23110: "Impedance arm (L)",
    23111: "Leg fat % (R)", 23112: "Leg fat mass (R)",
    23113: "Leg fat-free mass (R)", 23114: "Leg predicted mass (R)",
    23115: "Leg fat % (L)", 23116: "Leg fat mass (L)",
    23117: "Leg fat-free mass (L)", 23118: "Leg predicted mass (L)",
    23119: "Arm fat % (R)", 23120: "Arm fat mass (R)",
    23121: "Arm fat-free mass (R)", 23122: "Arm predicted mass (R)",
    23123: "Arm fat % (L)", 23124: "Arm fat mass (L)",
    23125: "Arm fat-free mass (L)", 23126: "Arm predicted mass (L)",
    23127: "Trunk fat %", 23128: "Trunk fat mass",
    23129: "Trunk fat-free mass", 23130: "Trunk predicted mass",
}

# Device/QC/skip-reason fields to exclude from analysis
EXCLUDE_FIELDS = {
    21, 36, 37, 39, 40, 41, 43, 44,   # device IDs
    96, 3077, 4081,                    # timing, box height, BP method
    20041, 20046, 20047, 20048,        # reason for skipping
}

print("Configuration ready.")


## 2. Data Loading and Initial Inspection

In [ ]:
def parse_col(col):
    m = re.match(r"^f_(\d+)_(\d+)", col)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

def is_baseline(col):
    if col == "eid":
        return True
    fid, inst = parse_col(col)
    if fid is None or fid in EXCLUDE_FIELDS:
        return False
    return inst == 0

def load_baseline(mod_name, info):
    path = DATA_ROOT / info["subdir"] / info["data"]
    if not path.exists():
        print(f"  [MISSING] {path}")
        return None
    print(f"  Loading {mod_name}...", end=" ")
    if info["fmt"] == "tsv":
        df = pd.read_csv(path, sep="\t", low_memory=False, encoding="latin-1")
    else:
        df = pd.read_parquet(path)
    baseline_cols = [c for c in df.columns if is_baseline(c)]
    df = df[baseline_cols].copy()
    print(f"{len(df):,} rows × {len(df.columns)} cols")
    return df

def eids_with_data(df):
    data_cols = [c for c in df.columns if c != "eid"]
    return set(df.loc[df[data_cols].notna().any(axis=1), "eid"].astype(int))

print("Loading all modalities (baseline instance only)...")
dfs, eid_sets = {}, {}
for mod, info in MODALITIES.items():
    df = load_baseline(mod, info)
    if df is not None:
        dfs[mod] = df
        eid_sets[mod] = eids_with_data(df)

print(f"\nLoaded {len(dfs)} modalities successfully.")


In [ ]:
# ── Initial inspection table ─────────────────────────────────────────────────
rows = []
for mod in MODALITIES:
    if mod not in dfs:
        continue
    df   = dfs[mod]
    es   = eid_sets[mod]
    dcols = [c for c in df.columns if c != "eid"]
    rows.append({
        "Modality":           LABELS[mod],
        "Baseline fields":    len(dcols),
        "Total rows":         f"{len(df):,}",
        "With data (N)":      f"{len(es):,}",
        "Mean missing (%)":   f"{df[dcols].isna().mean().mean()*100:.1f}",
        "Fields >40% miss":   (df[dcols].isna().mean() > 0.4).sum(),
    })

summary_df = pd.DataFrame(rows).set_index("Modality")
print("Initial data inspection:")
display(summary_df)


## 3. Statistical Summaries and Distributions

### 3.1 Age at Recruitment

In [ ]:
age = dfs["popchar"]["f_21022_0_0"].dropna().astype(float)

print(f"N participants with age : {len(age):,}")
print(f"Mean ± SD               : {age.mean():.1f} ± {age.std():.1f} years")
print(f"Median (IQR)            : {age.median():.1f}  ({age.quantile(0.25):.1f} – {age.quantile(0.75):.1f})")
print(f"Range                   : {age.min():.0f} – {age.max():.0f} years")
print()

# Decile table
deciles = age.quantile([0.1*i for i in range(1,10)])
deciles.index = [f"P{int(p*100)}" for p in deciles.index]
display(deciles.rename("Age (years)").to_frame().T.round(1))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(age, bins=36, color=PALETTE["popchar"], alpha=0.55,
        edgecolor="white", linewidth=0.5, density=True)
kde = gaussian_kde(age, bw_method=0.15)
x   = np.linspace(age.min(), age.max(), 500)
ax.plot(x, kde(x), color=PALETTE["popchar"], linewidth=2.5)
ax.axvline(age.mean(),   color="black", linestyle="--", lw=1.3,
           label=f"Mean = {age.mean():.1f}")
ax.axvline(age.median(), color="grey",  linestyle=":",  lw=1.3,
           label=f"Median = {age.median():.1f}")
ax.set_xlabel("Age at recruitment (years)")
ax.set_ylabel("Density")
ax.set_title(f"Age Distribution at Recruitment  (N = {len(age):,})")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_age_distribution.png")
plt.show()


### 3.2 Sex Distribution

In [ ]:
pc    = dfs["popchar"][["eid","f_21022_0_0","f_31_0_0"]].dropna()
n_f   = (pc["f_31_0_0"] == 0).sum()
n_m   = (pc["f_31_0_0"] == 1).sum()
total = len(pc)

print(f"Female : {n_f:,}  ({n_f/total*100:.1f}%)")
print(f"Male   : {n_m:,}  ({n_m/total*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].pie([n_f, n_m],
            labels=[f"Female\n{n_f:,} ({n_f/total*100:.1f}%)",
                    f"Male\n{n_m:,} ({n_m/total*100:.1f}%)"],
            colors=["#E07B8A","#5B8DB8"], startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":2.5})
axes[0].set_title(f"Sex Distribution  (N={total:,})")

for val, label, color in [(0,"Female","#E07B8A"),(1,"Male","#5B8DB8")]:
    ages = pc.loc[pc["f_31_0_0"]==val, "f_21022_0_0"].values
    kde  = gaussian_kde(ages, bw_method=0.15)
    x    = np.linspace(pc["f_21022_0_0"].min(), pc["f_21022_0_0"].max(), 400)
    axes[1].plot(x, kde(x), color=color, linewidth=2.5,
                 label=f"{label} (n={len(ages):,})")
    axes[1].fill_between(x, kde(x), alpha=0.15, color=color)
axes[1].set_xlabel("Age at recruitment (years)")
axes[1].set_ylabel("Density")
axes[1].set_title("Age Distribution by Sex")
axes[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_sex_distribution.png")
plt.show()


### 3.3 Sample Size per Modality

In [ ]:
# Exclude popchar from modality bar chart — it's the target source, not a feature modality
plot_mods = [m for m in MODALITIES if m != "popchar"]
ns     = [len(eid_sets.get(m, set())) for m in plot_mods]
labels = [LABELS[m] for m in plot_mods]
colors = [PALETTE[m] for m in plot_mods]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(labels, ns, color=colors, edgecolor="white", height=0.6)
for bar, n in zip(bars, ns):
    ax.text(bar.get_width() + 2000, bar.get_y() + bar.get_height()/2,
            f"{n:,}", va="center", fontsize=10)
ax.set_xlabel("Participants with baseline data")
ax.set_title("Sample Size per Feature Modality")
ax.set_xlim(0, max(ns) * 1.18)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_n_per_modality.png")
plt.show()


## 4. Missing Value Analysis

### 4.1 Per-field Missingness Within Modality Participants

In [ ]:
# Missingness computed WITHIN each modality's actual participant set
plot_mods = list(dfs.keys())
ncols = 3
nrows = int(np.ceil(len(plot_mods) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for ax, mod in zip(axes, plot_mods):
    df        = dfs[mod]
    eids      = np.array(list(eid_sets[mod]), dtype=int)
    df_sub    = df[df["eid"].isin(eids)]
    data_cols = [c for c in df_sub.columns if c != "eid"]
    miss_pct  = df_sub[data_cols].isna().mean() * 100

    ax.barh(range(len(miss_pct)), miss_pct.sort_values().values,
            color=PALETTE.get(mod,"#999"), alpha=0.75, height=1.0)
    ax.axvline(40, color="red", linestyle="--", lw=0.9, alpha=0.7,
               label="40% threshold")
    ax.set_title(
        f"{LABELS[mod]}\n"
        f"{len(data_cols)} fields | n={len(df_sub):,} | mean={miss_pct.mean():.1f}%",
        fontsize=9)
    ax.set_yticks([])
    ax.set_xlim(0, 100)
    ax.set_xlabel("% missing")
    ax.legend(fontsize=7, frameon=False)

for ax in axes[len(plot_mods):]:
    ax.set_visible(False)

fig.suptitle("Per-field Missingness at Baseline\n(within modality participants)",
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_missingness.png")
plt.show()


### 4.2 Within-modality Completeness Score

In [ ]:
# Per-participant: what fraction of fields are non-null?
score_mods = [m for m in dfs if m != "popchar"]
ncols = 3
nrows = int(np.ceil(len(score_mods) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

completeness_summary = []

for ax, mod in zip(axes, score_mods):
    df        = dfs[mod]
    eids      = np.array(list(eid_sets[mod]), dtype=int)
    df_sub    = df[df["eid"].isin(eids)]
    data_cols = [c for c in df_sub.columns if c != "eid"]
    comp      = df_sub[data_cols].notna().mean(axis=1) * 100

    ax.hist(comp, bins=50, color=PALETTE.get(mod,"#999"),
            alpha=0.8, edgecolor="white", linewidth=0.3)
    ax.axvline(comp.mean(),   color="black", ls="--", lw=1.2,
               label=f"Mean={comp.mean():.1f}%")
    ax.axvline(comp.median(), color="grey",  ls=":",  lw=1.2,
               label=f"Median={comp.median():.1f}%")

    pct_full = (comp == 100).mean() * 100
    pct_gt90 = (comp >=  90).mean() * 100
    ax.set_title(
        f"{LABELS[mod]}\n100% complete: {pct_full:.1f}%  |  ≥90%: {pct_gt90:.1f}%",
        fontsize=9)
    ax.set_xlabel("% of fields complete per participant")
    ax.set_ylabel("Count")
    ax.legend(fontsize=7, frameon=False)

    completeness_summary.append({
        "Modality":         LABELS[mod],
        "N participants":   len(df_sub),
        "Mean complete (%)":round(comp.mean(), 1),
        "Median (%)":       round(comp.median(), 1),
        "100% complete (%)":round(pct_full, 1),
        "≥90% complete (%)":round(pct_gt90, 1),
    })

for ax in axes[len(score_mods):]:
    ax.set_visible(False)

fig.suptitle("Within-modality Completeness Score per Participant",
             fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_completeness.png")
plt.show()

print("\nCompleteness summary:")
display(pd.DataFrame(completeness_summary).set_index("Modality"))


## 5. Cross-modality Overlap

In [ ]:
mods   = [m for m in eid_sets if m != "popchar"]
labels = [LABELS[m] for m in mods]
n      = len(mods)
raw    = np.zeros((n, n), dtype=int)
for i, a in enumerate(mods):
    for j, b in enumerate(mods):
        raw[i, j] = len(eid_sets[a] & eid_sets[b])

norm = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        denom = min(raw[i,i], raw[j,j])
        norm[i,j] = raw[i,j] / denom if denom > 0 else 0

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cmap_div = sns.diverging_palette(220, 20, as_cmap=True)

sns.heatmap(raw/1000, ax=axes[0], cmap="Blues", annot=True, fmt=".0f",
            xticklabels=labels, yticklabels=labels,
            cbar_kws={"label":"Participants (thousands)"}, linewidths=0.5)
axes[0].set_title("Cross-modality Overlap (N × 1000)")
plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right", fontsize=9)
plt.setp(axes[0].get_yticklabels(), rotation=0, fontsize=9)

sns.heatmap(norm, ax=axes[1], cmap="Greens", annot=True, fmt=".2f",
            xticklabels=labels, yticklabels=labels, vmin=0, vmax=1,
            cbar_kws={"label":"Fraction of smaller modality"}, linewidths=0.5)
axes[1].set_title("Cross-modality Overlap (Normalised)")
plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right", fontsize=9)
plt.setp(axes[1].get_yticklabels(), rotation=0, fontsize=9)

fig.tight_layout()
fig.savefig(OUT_DIR / "fig_overlap_heatmap.png")
plt.show()


In [ ]:
# Multi-modality participant counts
all_eids = set.union(*[eid_sets[m] for m in mods])
counts   = pd.Series({eid: sum(eid in eid_sets[m] for m in mods) for eid in all_eids})

print("Participants with data in N feature modalities simultaneously:\n")
rows = []
for n_mods in range(1, len(mods)+1):
    nn = (counts == n_mods).sum()
    if nn > 0:
        rows.append({"N modalities": n_mods, "Participants": nn,
                     "Cumulative (≥N)": (counts >= n_mods).sum()})
        print(f"  Exactly {n_mods}/{len(mods)} modalities : {nn:>8,}   "
              f"(cumulative ≥{n_mods}: {(counts >= n_mods).sum():,})")

display(pd.DataFrame(rows).set_index("N modalities"))


In [ ]:
if HAS_UPSET:
    all_eids_upset = set.union(*[eid_sets[m] for m in mods])
    memberships = [
        tuple(m for m in mods if eid in eid_sets[m])
        for eid in all_eids_upset
    ]
    data = from_memberships(memberships)
    fig  = plt.figure(figsize=(12, 6))
    UpSet(data, subset_size="count", show_counts=True,
          sort_by="cardinality", min_subset_size=500).plot(fig)
    fig.suptitle("Modality Combination Overlap (UpSet Plot)", y=1.01, fontsize=13)
    fig.savefig(OUT_DIR / "fig_upset.png")
    plt.show()
else:
    print("Install upsetplot to see UpSet plot.")


## 6. Feature Correlation Analysis

Within-modality Pearson correlation matrices, computed only over participants
who have actual data for that modality.

- Small modalities (blood count, biochemistry, physical measures): full
  clustered heatmap with hierarchical ordering.
- Metabolomics (253 features): clustered heatmap, smaller font.
- Olink (1,983 features): distribution of pairwise |r| values + top variable
  protein heatmap (1983×1983 full heatmap is visually meaningless).


In [ ]:
def load_chars_tsv(subdir, fname):
    path = DATA_ROOT / subdir / fname
    if not path.exists():
        return {}
    try:
        df = pd.read_csv(path, sep="\t", comment="#",
                         low_memory=False, encoding="latin-1")
        if "FieldID" in df.columns and "Field" in df.columns:
            return dict(zip(df["FieldID"].astype(int), df["Field"]))
    except Exception:
        pass
    return {}

FIELD_NAMES = {
    "physical_measures":  PHYS_FIELD_NAMES,
    "blood_biochemistry": load_chars_tsv("2_blood_measures","BloodBiochemistry_chars.tsv"),
    "blood_count":        load_chars_tsv("2_blood_measures","BloodCount_chars.tsv"),
    "metabolomics":       load_chars_tsv("2_blood_measures","metabolomics_chars.tsv"),
    "olink":              load_chars_tsv("2_blood_measures","olink_chars.tsv"),
}
for mod, fmap in FIELD_NAMES.items():
    print(f"  {LABELS[mod]:<35} {len(fmap)} field names loaded")


In [ ]:
def get_label(col, field_names, max_len=28):
    fid, _ = parse_col(col)
    name   = field_names.get(fid, col) if fid else col
    return name[:max_len-1] + "…" if len(name) > max_len else name

def drop_high_missing(df, thresh=0.4):
    data_cols = [c for c in df.columns if c != "eid"]
    keep = df[data_cols].columns[df[data_cols].isna().mean() <= thresh]
    return df[["eid"] + list(keep)]

def restrict(df, eid_set):
    return df[df["eid"].isin(np.array(list(eid_set), dtype=int))].copy()

def compute_corr(df):
    data = df[[c for c in df.columns if c != "eid"]].apply(pd.to_numeric, errors="coerce")
    return data.corr(method="pearson")

def cluster_order(corr):
    mat  = corr.fillna(0).values
    dist = np.clip(1 - np.abs(mat), 0, None)
    np.fill_diagonal(dist, 0)
    lnk  = sch.linkage(sch.distance.squareform(dist), method="ward")
    return sch.leaves_list(lnk)

def clustered_heatmap(corr, labels, title, out_path, figsize=None, fs=7):
    order  = cluster_order(corr)
    corr_r = corr.iloc[order, order]
    labs_r = [labels[i] for i in order]
    n = len(labs_r)
    if figsize is None:
        s = max(8, n * 0.28)
        figsize = (s, s * 0.85)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(corr_r, ax=ax,
                cmap=sns.diverging_palette(220, 20, as_cmap=True),
                center=0, vmin=-1, vmax=1, square=True, linewidths=0,
                xticklabels=labs_r, yticklabels=labs_r,
                cbar_kws={"label":"Pearson r","shrink":0.6})
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=fs)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=fs)
    ax.set_title(title, pad=12, fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path)
    plt.show()

print("Correlation helpers ready.")


### 6.1 Blood Count

In [ ]:
mod    = "blood_count"
df_sub = drop_high_missing(restrict(dfs[mod], eid_sets[mod]))
dcols  = [c for c in df_sub.columns if c != "eid"]
corr   = compute_corr(df_sub)
labs   = [get_label(c, FIELD_NAMES[mod]) for c in dcols]
print(f"Blood Count: {len(dcols)} features, {len(df_sub):,} participants")
clustered_heatmap(corr, labs,
    "Blood Count — Feature Correlation (Pearson, clustered)",
    OUT_DIR / "corr_blood_count.png")


### 6.2 Blood Biochemistry

In [ ]:
mod    = "blood_biochemistry"
df_sub = drop_high_missing(restrict(dfs[mod], eid_sets[mod]))
dcols  = [c for c in df_sub.columns if c != "eid"]
corr   = compute_corr(df_sub)
labs   = [get_label(c, FIELD_NAMES[mod]) for c in dcols]
print(f"Blood Biochemistry: {len(dcols)} features, {len(df_sub):,} participants")
clustered_heatmap(corr, labs,
    "Blood Biochemistry — Feature Correlation (Pearson, clustered)",
    OUT_DIR / "corr_blood_biochemistry.png")


### 6.3 Physical Measures

In [ ]:
mod    = "physical_measures"
df_sub = drop_high_missing(restrict(dfs[mod], eid_sets[mod]))
dcols  = [c for c in df_sub.columns if c != "eid"]
corr   = compute_corr(df_sub)
labs   = [get_label(c, FIELD_NAMES[mod]) for c in dcols]
print(f"Physical Measures: {len(dcols)} features, {len(df_sub):,} participants")
clustered_heatmap(corr, labs,
    "Physical Measures — Feature Correlation (Pearson, clustered)",
    OUT_DIR / "corr_physical_measures.png")


### 6.4 Metabolomics (NMR)

In [ ]:
mod    = "metabolomics"
df_sub = drop_high_missing(restrict(dfs[mod], eid_sets[mod]))
dcols  = [c for c in df_sub.columns if c != "eid"]
corr   = compute_corr(df_sub)
labs   = [get_label(c, FIELD_NAMES[mod], max_len=20) for c in dcols]
print(f"Metabolomics: {len(dcols)} features, {len(df_sub):,} participants")
clustered_heatmap(corr, labs,
    "Metabolomics (NMR) — Feature Correlation (Pearson, clustered)",
    OUT_DIR / "corr_metabolomics.png",
    figsize=(22, 18), fs=4.5)


### 6.5 Olink Proteomics

In [ ]:
mod    = "olink"
df_sub = drop_high_missing(restrict(dfs[mod], eid_sets[mod]))
dcols  = [c for c in df_sub.columns if c != "eid"]
print(f"Olink: {len(dcols)} features, {len(df_sub):,} participants")
print("Computing full correlation matrix (may take ~1-2 min)...")
corr   = compute_corr(df_sub)
print("Done.")


In [ ]:
# Distribution of pairwise |r|
vals = corr.values
mask = np.triu(np.ones_like(vals, dtype=bool), k=1)
r    = vals[mask]
r    = r[~np.isnan(r)]
absr = np.abs(r)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(r, bins=100, color=PALETTE["olink"], alpha=0.8,
             edgecolor="white", linewidth=0.2)
axes[0].axvline(0, color="black", lw=0.8, ls="--")
axes[0].set_xlabel("Pearson r")
axes[0].set_ylabel("Pair count")
axes[0].set_title("Pairwise Correlations — All Olink Proteins")
axes[0].text(0.97, 0.95,
    f"N pairs = {len(r):,}\nMean |r| = {absr.mean():.3f}\n"
    f"|r|>0.5: {(absr>0.5).mean()*100:.1f}%\n|r|>0.8: {(absr>0.8).mean()*100:.1f}%",
    transform=axes[0].transAxes, ha="right", va="top", fontsize=9,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))

axes[1].hist(absr, bins=100, color=PALETTE["olink"], alpha=0.8,
             edgecolor="white", linewidth=0.2, cumulative=True, density=True)
axes[1].axvline(0.5, color="red",  ls="--", lw=0.9, label="|r|=0.5")
axes[1].axvline(0.8, color="navy", ls="--", lw=0.9, label="|r|=0.8")
axes[1].set_xlabel("|Pearson r|")
axes[1].set_ylabel("Cumulative fraction of pairs")
axes[1].set_title("Cumulative |r| Distribution — Olink")
axes[1].legend(frameon=False)

fig.suptitle("Olink Proteomics — Pairwise Correlation Summary", fontsize=12)
fig.tight_layout()
fig.savefig(OUT_DIR / "corr_olink_distribution.png")
plt.show()


In [ ]:
# Top 50 most variable proteins — clustered heatmap
data_only = df_sub[[c for c in df_sub.columns if c != "eid"]]
top_cols  = data_only.var().nlargest(50).index.tolist()
corr_top  = corr.loc[top_cols, top_cols]
labs_top  = [get_label(c, FIELD_NAMES[mod], max_len=22) for c in top_cols]

clustered_heatmap(corr_top, labs_top,
    "Olink — Top 50 Most Variable Proteins (clustered heatmap)",
    OUT_DIR / "corr_olink_top50.png",
    figsize=(16, 14), fs=6)


In [ ]:
# Save top correlated pairs
rows = []
col_list = list(corr.columns)
corr_v   = corr.values
for i in range(len(col_list)):
    for j in range(i+1, len(col_list)):
        rv = corr_v[i, j]
        if not np.isnan(rv):
            fi, _ = parse_col(col_list[i])
            fj, _ = parse_col(col_list[j])
            rows.append({
                "protein_a": FIELD_NAMES["olink"].get(fi, col_list[i]) if fi else col_list[i],
                "protein_b": FIELD_NAMES["olink"].get(fj, col_list[j]) if fj else col_list[j],
                "pearson_r": round(rv, 4),
                "abs_r":     round(abs(rv), 4),
            })
top_pairs = (pd.DataFrame(rows)
             .sort_values("abs_r", ascending=False)
             .reset_index(drop=True))
top_pairs.to_csv(OUT_DIR / "olink_top_pairs.csv", index=False)
print(f"Saved {len(top_pairs):,} pairs to olink_top_pairs.csv")
print("\nTop 20 most correlated Olink protein pairs:")
display(top_pairs.head(20))


## 7. Data Quality Assessment

In [ ]:
# Per-modality quality summary
quality_rows = []
for mod in [m for m in MODALITIES if m != "popchar"]:
    if mod not in dfs:
        continue
    df        = dfs[mod]
    eids      = np.array(list(eid_sets[mod]), dtype=int)
    df_sub    = df[df["eid"].isin(eids)]
    data_cols = [c for c in df_sub.columns if c != "eid"]

    miss      = df_sub[data_cols].isna().mean() * 100
    comp      = df_sub[data_cols].notna().mean(axis=1) * 100

    # Variance — flag near-zero variance columns
    variance  = df_sub[data_cols].apply(pd.to_numeric, errors="coerce").var()
    low_var   = (variance < 1e-6).sum()

    quality_rows.append({
        "Modality":             LABELS[mod],
        "N participants":       f"{len(df_sub):,}",
        "N features (baseline)":len(data_cols),
        "Mean field miss (%)":  round(miss.mean(), 1),
        "Fields >40% miss":     (miss > 40).sum(),
        "Near-zero variance":   low_var,
        "Participants 100% complete (%)": round((comp==100).mean()*100, 1),
        "Participants ≥90% complete (%)": round((comp>=90).mean()*100, 1),
    })

quality_df = pd.DataFrame(quality_rows).set_index("Modality")
print("Data Quality Summary:")
display(quality_df)


In [ ]:
# Flag highly correlated feature pairs (potential redundancy) for small modalities
print("Highly correlated feature pairs (|r| > 0.95) — potential redundancy:\n")
for mod in ["blood_count", "blood_biochemistry", "physical_measures"]:
    if mod not in dfs:
        continue
    df_sub    = drop_high_missing(restrict(dfs[mod], eid_sets[mod]))
    data_cols = [c for c in df_sub.columns if c != "eid"]
    corr      = compute_corr(df_sub)
    corr_v    = corr.values
    fn        = FIELD_NAMES[mod]

    pairs = []
    for i in range(len(data_cols)):
        for j in range(i+1, len(data_cols)):
            rv = corr_v[i, j]
            if not np.isnan(rv) and abs(rv) > 0.95:
                fi, _ = parse_col(data_cols[i])
                fj, _ = parse_col(data_cols[j])
                pairs.append((fn.get(fi, data_cols[i]) if fi else data_cols[i],
                              fn.get(fj, data_cols[j]) if fj else data_cols[j],
                              round(rv, 3)))

    print(f"  {LABELS[mod]}: {len(pairs)} pairs with |r| > 0.95")
    for a, b, r in sorted(pairs, key=lambda x: -abs(x[2]))[:10]:
        print(f"    r={r:+.3f}  {a[:35]}  ↔  {b[:35]}")
    print()


## 8. Summary

In [ ]:
print("=" * 65)
print("  UK BIOBANK — EDA SUMMARY")
print("=" * 65)
print(f"\nTarget variable  : Age at recruitment")
print(f"  N              : {len(dfs['popchar']['f_21022_0_0'].dropna()):,}")
print(f"  Mean ± SD      : {dfs['popchar']['f_21022_0_0'].mean():.1f} ± "
      f"{dfs['popchar']['f_21022_0_0'].std():.1f} years")
print(f"  Range          : {dfs['popchar']['f_21022_0_0'].min():.0f} – "
      f"{dfs['popchar']['f_21022_0_0'].max():.0f}")

print("\nFeature modalities:")
for mod in [m for m in MODALITIES if m != "popchar"]:
    if mod in eid_sets:
        df_sub    = dfs[mod][dfs[mod]["eid"].isin(np.array(list(eid_sets[mod]),dtype=int))]
        data_cols = [c for c in df_sub.columns if c != "eid"]
        miss      = df_sub[data_cols].isna().mean().mean() * 100
        print(f"  {LABELS[mod]:<35} N={len(eid_sets[mod]):>7,}  "
              f"features={len(data_cols):>4}  mean_miss={miss:.1f}%")

print("\nModel cohort sizes:")
feat_mods  = [m for m in MODALITIES if m != "popchar"]
all_e      = set.union(*[eid_sets[m] for m in feat_mods])
counts_all = pd.Series({e: sum(e in eid_sets[m] for m in feat_mods) for e in all_e})
print(f"  All 5 feature modalities (inc. olink)  : "
      f"{(counts_all == 5).sum():>7,}")
print(f"  All 4 modalities excl. olink           : "
      f"{(counts_all >= 4).sum():>7,}")
print(f"  ≥3 modalities                          : "
      f"{(counts_all >= 3).sum():>7,}")

print("\nFigures saved to:", OUT_DIR.resolve())
